In [ ]:
import bnlearn as bn
from causallearn.search.ConstraintBased.PC import pc

# load ALARM
alarm_model = bn.import_DAG('alarm')
df_alarm = bn.sampling(alarm_model, n = 5000, methodtype='bayes')
#print(df_alarm)

data = df_alarm.to_numpy()
print('data:', data)

cg = pc(data, indep_test='chisq', node_names=list(df_alarm.columns))

In [ ]:
# Build true graph

from causallearn.graph.GeneralGraph import GeneralGraph
from causallearn.graph.GraphNode import GraphNode
from causallearn.graph.Edge import Edge
from causallearn.graph.Endpoint import Endpoint

# adj matrix
adj_mat = alarm_model['adjmat']

# create node objects same name with node_names used in pc
node_names = list(df_alarm.columns)
nodes = [GraphNode(name) for name in node_names]
node_dict = {n.get_name(): n for n in nodes}
truth_graph = GeneralGraph(nodes)
print("Adj matrix:", adj_mat)

for source in adj_mat.index:
    for target in adj_mat.columns:
        if adj_mat.loc[source, target] == 1:
            edge = Edge(node_dict[source], node_dict[target], Endpoint.TAIL, Endpoint.ARROW)
            truth_graph.add_edge(edge)

In [ ]:
# calculate SHD
# print(set(node_names) == set(adj_mat.index))

from causallearn.graph.SHD import SHD
shd = SHD(truth_graph, cg.G).get_shd()
print('SHD: ', shd)

### print
#true
print(alarm_model.keys())
print(type(alarm_model['model']))
true_model = alarm_model['model']

print("Number of nodes:", len(true_model.nodes()))

#for node in true_model.nodes():
#    print(node)
true_edges = list(true_model.edges())
print("Number of true edge:", len(true_edges))
for edge in true_edges:
    print(edge)

# learn
print("Number of learn edges:", len(cg.G.get_graph_edges()))
for edge in cg.G.get_graph_edges():
    print(edge)